In [12]:
from src.cyk import CYK
import pandas as pd
import numpy as np
import spacy


In [13]:
data = pd.read_csv("data/sentences.csv", sep=";")
G = [
        # S is axiom
        ("S", ("NP", "VP")), 
        
            # non terminal rules
        ("NP", ("DET", "NOUN")),
        ("PP", ("ADP", "NP")),
        ("VP", ("VERB", "PP")),

        
        # terminal (if using pos directly)
        ("DET", ("DET",)),
        ("VERB", ("VERB",)),
        ("NOUN", ("NOUN",)),
        ("ADP", ("ADP",))
    ]
    
cyk = CYK(G)

for _, example in data.iterrows():
    sentence = list(example['Sentence'].split())
    list_pos = []
    for p in example['Pos'].split():
        list_pos.append([p])
    print(f"Sentence: {sentence}")
    print(f"Pos: {list_pos}")
    
    tree = cyk(list_pos, sentence)
    for t in tree:
        print(t)
    print("##################################################################################")

    

Sentence: ['The', 'cat', 'sat', 'on', 'the', 'couch']
Pos: [['DET'], ['NOUN'], ['VERB'], ['ADP'], ['DET'], ['NOUN']]
S
----NP
--------DET - The
--------NOUN - cat
----VP
--------VERB - sat
--------PP
------------ADP - on
------------NP
----------------DET - the
----------------NOUN - couch


##################################################################################
Sentence: ['Time', 'flies', 'like', 'an', 'arrow']
Pos: [['NOUN'], ['NOUN'], ['ADP'], ['DET'], ['NOUN']]
##################################################################################
Sentence: ['The', 'spy', 'saw', 'the', 'cop', 'with', 'the', 'telescope']
Pos: [['DET'], ['NOUN'], ['VERB'], ['DET'], ['NOUN'], ['ADP'], ['DET'], ['NOUN']]
##################################################################################
Sentence: ['The', 'spy', 'saw', 'the', 'cop', 'with', 'the', 'revolver']
Pos: [['DET'], ['NOUN'], ['VERB'], ['DET'], ['NOUN'], ['ADP'], ['DET'], ['NOUN']]
##########################################

IndexError: list index out of range

# Automatic

In [25]:
nlp = spacy.load("en_core_web_sm")

Total_pos_list = []
for sentence in data['Sentence'].to_list():
    doc = nlp(sentence)
    print(f"Sentence: {sentence}")
    pos_list = [token.pos_ for token in doc]
    print(f"Pos: {pos_list}")
    Total_pos_list.append(pos_list)
    
print(f"Pos: {Total_pos_list}")

    



Sentence: The cat sat on the couch
Pos: ['DET', 'NOUN', 'VERB', 'ADP', 'DET', 'NOUN']
Sentence: Time flies like an arrow
Pos: ['NOUN', 'VERB', 'ADP', 'DET', 'NOUN']
Sentence: The spy saw the cop with the telescope
Pos: ['DET', 'NOUN', 'VERB', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN']
Sentence: The spy saw the cop with the revolver
Pos: ['DET', 'NOUN', 'VERB', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN']
Sentence: Arabidopsis thaliana seedlings exhibit longer hypocotyls when they are grown under high ambient temperature, which is defined as thermomorphogenesis
Pos: ['NOUN', 'NOUN', 'NOUN', 'VERB', 'ADV', 'NOUN', 'SCONJ', 'PRON', 'AUX', 'VERB', 'ADP', 'ADJ', 'ADJ', 'NOUN', 'PUNCT', 'PRON', 'AUX', 'VERB', 'ADP', 'NOUN']
Sentence: A spectrogram of PSN J10354824+3900279 obtained on Dec. 19.33 UT suggests that this is a type-Ia at redshift z 0.044
Pos: ['DET', 'NOUN', 'ADP', 'PROPN', 'PROPN', 'PROPN', 'NUM', 'VERB', 'ADP', 'PROPN', 'NUM', 'PROPN', 'VERB', 'SCONJ', 'PRON', 'AUX', 'DET', 'NOUN', 'PUNCT', '

In [31]:
class EvaluatePOS:
    def __init__(self, gold_pos, predicted_pos):
        self.gold_pos = gold_pos 
        self.predicted_pos = predicted_pos

    def accuracy(self):

        # Check same number of sentences
        if len(self.gold_pos) != len(self.predicted_pos):
            raise ValueError(
                "gold_pos and predicted_pos must have the same number of sentences."
            )

        # Check same number of POS tags for each sentence
        for i, (gold, predicted) in enumerate(
            zip(self.gold_pos, self.predicted_pos)
        ):
            if len(gold) != len(predicted):
                raise ValueError(
                    f"Sentence {i} has different lengths: "
                    f"gold={len(gold)}, predicted={len(predicted)}"
                )

        correct = 0
        total = 0

        for gold, predicted in zip(self.gold_pos, self.predicted_pos):
            for g, p in zip(gold, predicted):
                if g == p:
                    correct += 1
                total += 1

        return correct / total if total > 0 else 0.0



gold_pos = [example['Pos'].split() for _, example in data.iterrows()]
print(f"Gold POS: {gold_pos}")
print(f"Predicted POS: {Total_pos_list}")


EvaluatePOS(gold_pos, Total_pos_list).accuracy()

Gold POS: [['DET', 'NOUN', 'VERB', 'ADP', 'DET', 'NOUN'], ['NOUN', 'NOUN', 'ADP', 'DET', 'NOUN'], ['DET', 'NOUN', 'VERB', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN'], ['DET', 'NOUN', 'VERB', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN'], ['PROPN', 'PROPN', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'SCONJ', 'PRON', 'AUX', 'VERB', 'ADP', 'ADJ', 'ADJ', 'NOUN', 'PUNCT', 'PRON', 'AUX', 'VERB', 'ADP', 'NOUN', 'PUNCT'], ['DET', 'NOUN', 'ADP', 'NOUN', 'NOUN', 'VERB', 'ADP', 'NOUN', 'ADP', 'DET', 'VERB', 'DET', 'NOUN', 'ADP', 'NOUN']]
Predicted POS: [['DET', 'NOUN', 'VERB', 'ADP', 'DET', 'NOUN'], ['NOUN', 'VERB', 'ADP', 'DET', 'NOUN'], ['DET', 'NOUN', 'VERB', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN'], ['DET', 'NOUN', 'VERB', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN'], ['NOUN', 'NOUN', 'NOUN', 'VERB', 'ADV', 'NOUN', 'SCONJ', 'PRON', 'AUX', 'VERB', 'ADP', 'ADJ', 'ADJ', 'NOUN', 'PUNCT', 'PRON', 'AUX', 'VERB', 'ADP', 'NOUN'], ['DET', 'NOUN', 'ADP', 'PROPN', 'PROPN', 'PROPN', 'NUM', 'VERB', 'ADP', 'PROPN', 'NUM', 'PROPN', 'VERB', 'SCONJ

ValueError: Sentence 4 has different lengths: gold=21, predicted=20